# PySpark 4: windows and nested data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-04-windows-and-nested-data.ipynb)

A change feed sends several versions of each order, clickstream events arrive as nested JSON, and reports
need running totals in each customer's local time. None of these fit a plain `groupBy`.

This module uses window functions to rank rows, compare neighbours, compute running and moving totals, and
pick the latest record per key deterministically. It then handles dates, timestamps and time zones, and
flattens nested structs and arrays without losing rows.

Each section explains how the feature works, where it matters and what to watch out for, then ends with a runnable example. Run it, read the output, and compare it with the notes on what to notice. Run the cells in order.

**Prerequisites:** [PySpark 3: joins and aggregations](pyspark-03-joins-and-aggregations.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `orders` (one row per order; order 12 lands just after midnight UTC) and `events` (one row per user event; one event is an exact retry): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-foundations.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)),
    (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)),
    (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)),
    (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)),
    (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)),
    (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),
    (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

## How window functions work

A window function computes a value for each row from a set of related rows, without collapsing them the way
`groupBy` does. A window specification has three parts:

- `partitionBy(...)`: which rows belong together, for example one customer;
- `orderBy(...)`: their order inside the partition;
- a **frame** (`rowsBetween` or `rangeBetween`): which neighbours each row can see.

Spark plans a partitioned window as a shuffle by the partition key, a sort inside each partition, and one
pass that evaluates every window expression sharing that specification.

**Where it matters**
- **Latest record per key** in change data capture.
- **Rankings** within a segment, running totals, moving averages and gaps between events.

**Watch out for**
- A window with no `partitionBy` sends every row to a single partition and a single task.
- Every distinct window specification adds its own shuffle and sort. Reuse one specification for several expressions where you can.
- A hot partition key (one customer with 40 percent of rows) makes one slow task.

### Example

Keep each customer's most recent order with `row_number` over a window ordered by time, newest first.

In [ ]:
from pyspark.sql import Window

newest_first = Window.partitionBy("customer_id").orderBy(F.col("order_ts").desc(), F.col("order_id").desc())

latest_order = (
    orders
    .withColumn("rn", F.row_number().over(newest_first))
    .where("rn = 1")
    .drop("rn")
)
latest_order.orderBy("customer_id").select("customer_id", "order_id", "status", "order_ts").show()

**What to notice:** One row per customer remains, six in all, and every column of the winning order is kept: a `groupBy` with
`max(order_ts)` would have returned the timestamp only. Customer 1 placed three orders on 5 January; order 3
is the latest.

## Rank rows: row_number, rank and dense_rank

The three ranking functions differ only on ties:

| function | values for 300, 300, 200 |
| --- | --- |
| `row_number` | 1, 2, 3 (ties broken arbitrarily unless the order is unique) |
| `rank` | 1, 1, 3 (gaps after ties) |
| `dense_rank` | 1, 1, 2 (no gaps) |

`ntile(n)` splits a partition into `n` buckets, and `percent_rank` gives a 0-to-1 position.

**Where it matters**
- **Exactly one row per key:** `row_number` with a unique ordering.
- **Top N per group, keeping ties:** `dense_rank() <= N`.

**Watch out for**
- `row_number` over a non-unique order picks a different winner between runs. Add a tie-breaker column.
- "Top 2" is ambiguous with ties. Decide whether you want two rows (`row_number`) or two values (`dense_rank`).

### Example

Two north reps tie at 300. Compare the three functions on the same window.

In [ ]:
sales = spark.createDataFrame(
    [("north", "A", 300), ("north", "B", 300), ("north", "C", 200), ("south", "D", 500)],
    "region string, rep string, revenue int",
)
by_revenue = Window.partitionBy("region").orderBy(F.desc("revenue"))
ranked = sales.select(
    "region", "rep", "revenue",
    F.row_number().over(by_revenue).alias("row_number"),
    F.rank().over(by_revenue).alias("rank"),
    F.dense_rank().over(by_revenue).alias("dense_rank"),
)
ranked.orderBy("region", "row_number").show()

**What to notice:** A and B tie at 300. `row_number` still gives them 1 and 2, in an order you should not rely on. `rank` gives
both 1 and jumps to 3 for C; `dense_rank` gives both 1 and C gets 2.

## Compare neighbouring rows with lag and lead

`lag(col, n)` reads the value `n` rows earlier in the window order and `lead(col, n)` the value `n` rows
later; both return null (or a default you pass) at the edges of a partition. Combined with date functions
they give gaps between events, changes against the previous value, and the next step in a journey.

**Where it matters**
- **Days between orders** for churn and repeat-purchase metrics.
- **Status transitions** in a change feed: what was the previous status?

**Watch out for**
- The first row of each partition has no previous value. Decide whether null, zero or the row's own value is right.
- Without `partitionBy`, `lag` reads across customers and leaks one customer's data into another's metric.

### Example

For each order, find the previous order time and the next order id of the same customer.

In [ ]:
in_time_order = Window.partitionBy("customer_id").orderBy("order_ts", "order_id")
gaps = (
    orders.select(
        "customer_id", "order_id", "order_ts",
        F.lag("order_ts").over(in_time_order).alias("previous_ts"),
        F.lead("order_id").over(in_time_order).alias("next_order_id"),
    )
    .withColumn("days_since_previous", F.datediff(F.to_date("order_ts"), F.to_date("previous_ts")))
)
gaps.orderBy("customer_id", "order_ts").show()

**What to notice:** Each customer's first order has no previous timestamp, and the last has no next order: `lag` and `lead`
return null at the edges of the partition. Customer 2 reordered after one day, customer 3 after two.
Customer 1's three same-day orders show zero days between them.

## Running totals and moving windows: rows versus range frames

The frame decides which rows each calculation sees:

- `rowsBetween(Window.unboundedPreceding, Window.currentRow)`: a running total over physical rows;
- `rowsBetween(-2, 0)`: this row and the two before it, whatever their timestamps;
- `rangeBetween(-7 * 86400, 0)` over a numeric ordering: every row whose value is within seven days of this
  one, however many rows that is.

When a window has `orderBy` but no explicit frame, Spark uses a range frame from the start of the partition
to the current row. That is right for running totals, but it makes `last` return the current row rather
than the partition's last row.

**Where it matters**
- **Running revenue** per customer or per day.
- **Trailing 7-day spend** for fraud or loyalty rules, where calendar time matters more than row counts.

**Watch out for**
- `last` or `last_value` with the default frame returns the current row. Use an explicit frame to `unboundedFollowing`.
- A rows frame over irregular timestamps mixes a day's gap with a month's gap. Use a range frame for calendar windows.
- Range frames need a numeric or date ordering; cast a timestamp to `long` seconds as below.

### Example

Compute a running total and a trailing seven-day sum per customer, then show the default-frame trap:
`last("order_id")` returns each row's own id until the frame is widened.

In [ ]:
running = in_time_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)
seven_days = (
    Window.partitionBy("customer_id")
    .orderBy(F.col("order_ts").cast("long"))
    .rangeBetween(-7 * 86400, 0)
)
totals = orders.select(
    "customer_id", "order_id", "order_ts", "amount",
    F.sum("amount").over(running).alias("running_total"),
    F.sum("amount").over(seven_days).alias("spend_7d"),
)
totals.orderBy("customer_id", "order_ts").show()

whole_partition = in_time_order.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
last_ids = orders.where("customer_id = 1").select(
    "order_id",
    F.last("order_id").over(in_time_order).alias("default_frame"),
    F.last("order_id").over(whole_partition).alias("whole_partition"),
)
last_ids.orderBy("order_id").show()

**What to notice:** Customer 1's running total climbs 120.00, 204.50, 234.50. Order 5's seven-day spend is 315.25 because
order 4, a day earlier, is inside the range frame. In the second table, `default_frame` simply repeats each
row's own id: the default frame ends at the current row. Only the explicit whole-partition frame returns 3,
the real last order.

## Pick the latest record deterministically

Change data capture (CDC) feeds send several versions of the same key, and retries can deliver the same
change twice. Keep one row per key with `row_number` over a window ordered by the **source** sequence (a log
sequence number, a version, an `updated_at` from the source system), then a tie-breaker that is stable
across reruns.

`dropDuplicates(["order_id"])` also keeps one row per key, but it has no ordering: which row survives is
not defined.

**Where it matters**
- **Silver tables** built from a change feed: one current row per order.
- **Replays:** a reprocessed batch must pick the same winners as the first run.

**Watch out for**
- Ordering only by a timestamp with second precision. Two updates in the same second tie, and the winner can change after a repartition.
- Using processing or ingestion time as the main ordering. A replay changes it; use it only as a last tie-breaker.
- `dropDuplicates` with no arguments compares whole rows: right for exact retries, wrong for picking the latest version.

### Example

Order 101 has three versions; order 102's final change was replayed and ingested twice. Order by source
sequence, then by ingestion time, and keep the first row of each key.

In [ ]:
orders_cdc = spark.createDataFrame(
    [
        (101, "placed",    1, datetime(2026, 1, 5, 9, 0)),
        (101, "paid",      2, datetime(2026, 1, 5, 9, 5)),
        (101, "shipped",   3, datetime(2026, 1, 6, 8, 0)),
        (102, "placed",    1, datetime(2026, 1, 5, 10, 0)),
        (102, "cancelled", 2, datetime(2026, 1, 5, 10, 1)),
        (102, "cancelled", 2, datetime(2026, 1, 5, 10, 9)),  # replayed change, later ingest
    ],
    "order_id int, status string, source_lsn long, ingested_at timestamp",
)
latest_change = Window.partitionBy("order_id").orderBy(F.desc("source_lsn"), F.desc("ingested_at"))
current = orders_cdc.withColumn("rn", F.row_number().over(latest_change)).where("rn = 1").drop("rn")
current.orderBy("order_id").show()

exact_retries_removed = events.dropDuplicates()  # whole-row comparison: the SDK retry disappears
print("events:", events.count(), "| after dropDuplicates():", exact_retries_removed.count())

**What to notice:** Order 101 keeps its highest sequence, `shipped`. Order 102's final change arrived twice with the same
sequence; the ingestion time breaks the tie, so the 10:09 copy wins on every run. `dropDuplicates()` with no
arguments is the right tool only for exact retries, like the duplicated event.

## Dates, timestamps and time zones

A `TIMESTAMP` in Spark is an instant: it is stored in UTC and displayed in the session time zone
(`spark.sql.session.timeZone`). `TIMESTAMP_NTZ` is a wall-clock value with no time zone, shown the same
everywhere. `to_date`, `date_trunc`, `date_add`, `datediff`, `weekofyear` and friends derive calendar
fields; `from_utc_timestamp` converts an instant to a local wall-clock time for reporting.

**Where it matters**
- **Business dates:** an order at 00:30 UTC belongs to the previous day in Chicago.
- **Partition columns** derived from event time, such as `order_date`.

**Watch out for**
- Pin the session time zone in every job. A cluster default of `America/Los_Angeles` silently moves every derived date.
- Use `TIMESTAMP_NTZ` for values that are already local wall-clock times (a store's opening hours), and `TIMESTAMP` for instants.
- `current_date()` makes output depend on when the job ran, which breaks reruns. Pass the business date as a parameter.

### Example

Derive calendar fields, convert order 12 (00:30 UTC) to Chicago time, and compare how `TIMESTAMP` and
`TIMESTAMP_NTZ` display when the session time zone changes.

In [ ]:
with_dates = orders.select(
    "order_id",
    F.to_date("order_ts").alias("order_date"),
    F.date_trunc("month", "order_ts").alias("order_month"),
    F.weekofyear("order_ts").alias("order_week"),
    F.date_add(F.to_date("order_ts"), 2).alias("ship_by_date"),
    F.from_utc_timestamp("order_ts", "America/Chicago").alias("order_ts_chicago"),
)
with_dates.orderBy("order_id").show(truncate=False)

wall_clock = spark.createDataFrame(
    [(datetime(2026, 1, 14, 0, 30), datetime(2026, 1, 14, 0, 30))], "instant timestamp, local timestamp_ntz"
)
spark.conf.set("spark.sql.session.timeZone", "America/Chicago")
try:
    shown = wall_clock.select(F.col("instant").cast("string"), F.col("local").cast("string")).first()
finally:
    spark.conf.set("spark.sql.session.timeZone", "UTC")
print("shown with the session in America/Chicago -> instant:", shown[0], "| wall clock:", shown[1])

**What to notice:** Order 12 was placed at 00:30 UTC on 14 January, which is 18:30 on 13 January in Chicago: the business date
depends on the reporting time zone. In the last line, the `TIMESTAMP` value is displayed in the session's
time zone and moves to the 13th, while the `TIMESTAMP_NTZ` value stays at 00:30 on the 14th.

## Flatten nested structs and arrays

JSON and Parquet readers expose nested objects as `struct` columns and lists as `array` columns. Read a struct
field with a dotted path (`F.col("user.id")`). Turn an array into rows with `explode`, or with
`explode_outer`, which keeps a row (with nulls) when the array is null or empty. `posexplode` also returns
each element's position. `from_json` parses a JSON string column with a schema, and `F.struct` with
`to_json` builds nested output.

To work on an array without exploding it, use higher-order functions: `transform` maps each element,
`filter` keeps some, `aggregate` folds them into one value, and `exists` tests them.

**Where it matters**
- **Clickstream and API payloads** with nested device, user and item fields.
- **Order lines stored as an array** inside each order event.

**Watch out for**
- `explode` silently drops rows whose array is null or empty: a classic correctness bug. Use `explode_outer` unless dropping them is intended.
- Exploding multiplies rows. An average of four items per event turns 100 million events into 400 million rows for every later join and write.
- Parse JSON strings with an explicit schema so a missing field becomes null instead of an error or a schema change.

### Example: explode without losing rows

Land four events: `e3` has a null `items` array and `e4` an empty one. `explode` loses both;
`explode_outer` keeps both.

In [ ]:
import json

nested_dir = os.path.join(WORK_DIR, "raw", "events")
os.makedirs(nested_dir, exist_ok=True)
nested_rows = [
    {"event_id": "e1", "user": {"id": 1}, "device": {"os": "ios"},
     "items": [{"sku": "SKU-4001", "quantity": 2}, {"sku": "SKU-4007", "quantity": 1}],
     "payload_json": '{"source": "email", "campaign": {"id": "c-9", "name": "winter"}}'},
    {"event_id": "e2", "user": {"id": 2}, "device": {"os": "android"},
     "items": [{"sku": "SKU-4002", "quantity": 1}],
     "payload_json": '{"source": "ads", "campaign": {"id": "c-3", "name": "search"}}'},
    {"event_id": "e3", "user": {"id": 3}, "device": {"os": "web"}, "items": None,
     "payload_json": '{"source": "direct", "campaign": null}'},
    {"event_id": "e4", "user": {"id": 4}, "device": {"os": "web"}, "items": [],
     "payload_json": '{"source": "push"}'},
]
with open(os.path.join(nested_dir, "part-0.json"), "w") as handle:
    handle.write("\n".join(json.dumps(row) for row in nested_rows))

raw_events = spark.read.schema(
    "event_id string, user struct<id:int>, device struct<os:string>, "
    "items array<struct<sku:string,quantity:int>>, payload_json string"
).json(nested_dir)

items = (
    raw_events
    .select("event_id", F.col("user.id").alias("user_id"), F.explode_outer("items").alias("item"))
    .select("event_id", "user_id", F.col("item.sku").alias("sku"), F.col("item.quantity").alias("quantity"))
)
items.orderBy("event_id", "sku").show()
print("rows with explode:", raw_events.select(F.explode("items")).count(), "| with explode_outer:", items.count())

**What to notice:** `e3` (null array) and `e4` (empty array) each keep one row with null item fields. Plain `explode` returns
three rows and both events vanish, with no error to warn you.

### Example: parse JSON and work on arrays in place

Parse the JSON payload with a schema (a missing `campaign` becomes null), work on the array with
higher-order functions instead of exploding it, and build JSON output from a struct.

In [ ]:
parsed = raw_events.select(
    "event_id",
    F.from_json("payload_json", "source string, campaign struct<id:string,name:string>").alias("payload"),
).select("event_id", "payload.source", F.col("payload.campaign.id").alias("campaign_id"))
parsed.orderBy("event_id").show()

in_place = raw_events.select(
    "event_id",
    F.transform("items", lambda item: F.lower(item["sku"])).alias("skus"),
    F.filter("items", lambda item: item["quantity"] > 1).alias("multi_unit"),
    F.aggregate("items", F.lit(0), lambda total, item: total + item["quantity"]).alias("units"),
)
in_place.orderBy("event_id").show(truncate=False)

as_json = raw_events.select(
    "event_id", F.to_json(F.struct(F.col("user.id").alias("user_id"), F.col("device.os").alias("os"))).alias("json")
)
as_json.orderBy("event_id").show(truncate=False)

**What to notice:** `e4`'s payload has no `campaign`, and `e3`'s is null: both become a null `campaign_id` instead of an error.
The higher-order functions keep one row per event: `e1` has two SKUs, one line with more than one unit and
three units in total; `e4`'s empty array gives an empty list and 0 units; `e3`'s null array stays null.
`to_json(struct(...))` builds compact JSON output.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** What happens to a window function with no `partitionBy`?

**A.** Every row moves to one partition and one task, so the step stops being distributed. Spark logs a warning; on real data it is slow or runs out of memory.

**Q2.** Salaries 300, 300 and 200: what do `rank` and `dense_rank` return?

**A.** `rank` returns 1, 1, 3 and `dense_rank` returns 1, 1, 2. `row_number` returns 1, 2, 3 with an arbitrary order between the ties.

**Q3.** Why does `last("x")` over `orderBy(ts)` return the current row's value?

**A.** With `orderBy` and no frame, the default frame ends at the current row. Use `rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)` to see the whole partition.

**Q4.** When do you use `rangeBetween` instead of `rowsBetween`?

**A.** When the window is defined in calendar time (the last seven days) rather than a number of rows. A range frame includes every row within the interval, however many there are.

**Q5.** Why is `dropDuplicates(["order_id"])` wrong for keeping the latest change?

**A.** It has no ordering, so the surviving row is not defined and can change between runs. Use `row_number` over a window ordered by the source sequence with a tie-breaker.

**Q6.** An order at 00:30 UTC: which date does it belong to?

**A.** It depends on the reporting time zone: 14 January in UTC, 13 January in Chicago. Pin the session time zone and convert explicitly with `from_utc_timestamp`.

**Q7.** Which rows does `explode` drop, and what do you use instead?

**A.** Rows whose array is null or empty. `explode_outer` keeps them with null element fields.

## Key takeaways

- A window keeps every row; `partitionBy`, `orderBy` and the frame decide what each row can see.
- `row_number` over a source sequence plus a tie-breaker is the standard latest-record pattern.
- Know the default frame: running totals work, `last` does not.
- Pin the session time zone; use `TIMESTAMP_NTZ` only for true wall-clock values.
- Use `explode_outer` unless losing empty arrays is intended, and count rows before and after.

**Next:** [PySpark 5: writing, pipelines and quality](pyspark-05-writing-pipelines-and-quality.ipynb)